# 03 - Answering with Distractor as context

This notebook loads the constructed RAG uncertainty evaluation dataset, sends each query to **GPT-5** with only the **distractor passage** as context, and saves a new CSV containing all original columns plus `answer_with_distractor`.

The intended use is to test whether the model answers from an answer-free but topically relevant passage, abstains, or hedges.

## Setup

In [ ]:
from pathlib import Path
import os
import time
import pandas as pd
from tqdm.auto import tqdm

/Users/martasumyk/Desktop/Workshop/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
MODEL = "gpt-5"
TEMPERATURE = 0
MAX_RETRIES = 1
SLEEP_BETWEEN_CALLS = 0.2

## Load data

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from the notebook location or current working directory."""
    start = Path.cwd() if start is None else start
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and ((path / "data").exists() or (path / "notebooks").exists()):
            return path
    return start

ROOT = find_project_root()
DATA_DIR = ROOT / "data"

INPUT_CSV = None

candidate_files = [
    DATA_DIR / "rag_eval_en_full.csv",
    DATA_DIR / "rag_eval_en_300.csv",
    DATA_DIR / "rag_eval_en_full_with_difficulty.csv",
]

if INPUT_CSV is None:
    for candidate in candidate_files:
        if candidate.exists():
            INPUT_CSV = candidate
            break

if INPUT_CSV is None:
    csv_files = sorted(DATA_DIR.glob("*.csv")) if DATA_DIR.exists() else []
    INPUT_CSV = csv_files[0]

print(f"Project root: {ROOT}")
print(f"Loading: {INPUT_CSV}")

df = pd.read_csv(INPUT_CSV)
print(df.shape)
df.head()

Project root: /Users/martasumyk/Desktop/Workshop/rag-uncertainty-eval
Loading: /Users/martasumyk/Desktop/Workshop/rag-uncertainty-eval/data/rag_eval_en_full.csv
(6024, 17)


,query_id,query,distractor_id,distractor_text,distractor_title,gold_passage_id,gold_passage_text,gold_passage_title,gold_answer,n_positives,n_negatives,neg_rank,tfidf_sim,difficulty,query_len,distractor_len,gold_len
0,0,Is Creole a pidgin of French?,11236157#2,While many creoles around the world have lexic...,Monogenetic theory of pidgins,22399755#18,There are many hypotheses on the origins of Ha...,Haitian literature,Haitian literature,4,8,0,0.1469,medium,29,965,749
1,0,Is Creole a pidgin of French?,1170520#2,Louisiana Creole is a contact language that ar...,Louisiana Creole,22399755#18,There are many hypotheses on the origins of Ha...,Haitian literature,Haitian literature,4,8,1,0.1783,medium,29,974,749
2,0,Is Creole a pidgin of French?,1612877#8,"A mixed language differs from pidgins, creoles...",Mixed language,22399755#18,There are many hypotheses on the origins of Ha...,Haitian literature,Haitian literature,4,8,2,0.0853,easy,29,851,749
3,0,Is Creole a pidgin of French?,1651722#10,Krio is an English-based creole from which des...,Krio language,22399755#18,There are many hypotheses on the origins of Ha...,Haitian literature,Haitian literature,4,8,3,0.2026,hard,29,454,749
4,0,Is Creole a pidgin of French?,49823#1,The precise number of creole languages is not ...,Creole language,22399755#18,There are many hypotheses on the origins of Ha...,Haitian literature,Haitian literature,4,8,4,0.1654,medium,29,952,749


In [5]:
def pick_column(df: pd.DataFrame, candidates: list[str], description: str) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(
        f"Could not find {description}. Tried: {candidates}. Available columns: {list(df.columns)}"
    )

QUERY_COL = pick_column(
    df,
    ["query", "question", "query_text", "text"],
    "query/question column",
)

DISTRACTOR_COL = pick_column(
    df,
    ["distractor", "distractor_text", "negative_passage", "negative_text", "passage", "context"],
    "distractor/context column",
)

TITLE_COL = None
for candidate in ["distractor_title", "negative_title", "title"]:
    if candidate in df.columns:
        TITLE_COL = candidate
        break

print("QUERY_COL:", QUERY_COL)
print("DISTRACTOR_COL:", DISTRACTOR_COL)
print("TITLE_COL:", TITLE_COL)
print("Columns:", list(df.columns))

QUERY_COL: query
DISTRACTOR_COL: distractor_text
TITLE_COL: distractor_title
Columns: ['query_id', 'query', 'distractor_id', 'distractor_text', 'distractor_title', 'gold_passage_id', 'gold_passage_text', 'gold_passage_title', 'gold_answer', 'n_positives', 'n_negatives', 'neg_rank', 'tfidf_sim', 'difficulty', 'query_len', 'distractor_len', 'gold_len']


## Prompt and GPT-5 call

In [ ]:
# If the context does not contain enough information to answer the question, say: "I don't know based on the provided context."?

In [ ]:
SYSTEM_PROMPT = """You are a question-answering system for RAG evaluation.
Use only the provided context passage.
Do not use outside knowledge.
Keep the answer concise."""

In [7]:
def build_user_prompt(row: pd.Series) -> str:
    title_part = ""

    if TITLE_COL is not None and pd.notna(row.get(TITLE_COL)):
        title_part = f"Context title: {row[TITLE_COL]}\n\n"

    return f"""Question:
{row[QUERY_COL]}

{title_part}Context passage:
{row[DISTRACTOR_COL]}

Answer:"""


def extract_response_text(response) -> str:
    """Extract text from OpenAI Responses API output."""
    if hasattr(response, "output_text") and response.output_text:
        return response.output_text.strip()

    try:
        chunks = []

        for item in response.output:
            if getattr(item, "type", None) == "message":
                for content in item.content:
                    text = getattr(content, "text", None)
                    if text:
                        chunks.append(text)

        return "\n".join(chunks).strip()

    except Exception:
        return str(response)


def ask_gpt5_with_distractor(row: pd.Series) -> str:
    user_prompt = build_user_prompt(row)

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=MODEL,
                input=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=TEMPERATURE,
            )

            return extract_response_text(response)

        except Exception as e:
            last_error = e
            wait = min(2 ** attempt, 60)
            print(
                f"Attempt {attempt}/{MAX_RETRIES} failed: {e}. "
                f"Retrying in {wait}s..."
            )
            time.sleep(wait)

    return f"ERROR: {last_error}"

## Inference on dataset

In [ ]:
N_ROWS = None

OUTPUT_CSV = DATA_DIR / f"{Path(INPUT_CSV).stem}_with_gpt5_distractor_answers.csv"
ANSWER_COL = "answer_with_distractor"

if OUTPUT_CSV.exists():
    print(f"Found existing output, resuming from: {OUTPUT_CSV}")
    out_df = pd.read_csv(OUTPUT_CSV)
else:
    out_df = df.copy()
    if ANSWER_COL not in out_df.columns:
        out_df[ANSWER_COL] = pd.NA

if N_ROWS is not None:
    rows_to_process = out_df.head(N_ROWS).index
else:
    rows_to_process = out_df.index

missing_mask = out_df.loc[rows_to_process, ANSWER_COL].isna() | (out_df.loc[rows_to_process, ANSWER_COL].astype(str).str.strip() == "")
indices = list(out_df.loc[rows_to_process].loc[missing_mask].index)

In [ ]:
SAVE_EVERY = 10

for n, idx in enumerate(tqdm(indices), start=1):
    out_df.at[idx, ANSWER_COL] = ask_gpt5_with_distractor(out_df.loc[idx])
    time.sleep(SLEEP_BETWEEN_CALLS)

    if n % SAVE_EVERY == 0:
        out_df.to_csv(OUTPUT_CSV, index=False)
        print(f"Saved checkpoint after {n} new rows -> {OUTPUT_CSV}")

out_df.to_csv(OUTPUT_CSV, index=False)
print(f"Done. Saved: {OUTPUT_CSV}")
out_df.head()

## Samples

In [ ]:
preview_cols = [col for col in [QUERY_COL, TITLE_COL, DISTRACTOR_COL, ANSWER_COL] if col is not None]

pd.set_option("display.max_colwidth", 400)
out_df[preview_cols].head(10)